In [1]:
import xgboost
import catboost
import lightgbm
import pandas as pd
import numpy as np
import os
import sys
from dotenv import load_dotenv
import json
from sklearn.pipeline import Pipeline
import mlflow

In [2]:
load_dotenv()
data_path = os.getenv("DATA_PATH")
src_path = os.getenv("SRC_PATH")
sys.path.append(src_path)
from about_data.data_load import load_df

In [3]:
pd.set_option("display.max_columns", 100)
full_df = load_df(data_path)

In [4]:
full_df.head(2)

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,addr1,addr2,dist1,dist2,P_emaildomain,R_emaildomain,C1,C2,C3,C4,C5,C6,C7,C8,C9,C10,C11,C12,C13,C14,D1,D2,D3,D4,D5,D6,D7,D8,D9,D10,D11,D12,D13,D14,D15,M1,M2,M3,M4,...,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,id_10,id_11,id_12,id_13,id_14,id_15,id_16,id_17,id_18,id_19,id_20,id_21,id_22,id_23,id_24,id_25,id_26,id_27,id_28,id_29,id_30,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,credit,315.0,87.0,19.0,NaN,NaN,NaN,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,2.0,0.0,1.0,1.0,14.0,NaN,13.0,NaN,NaN,NaN,NaN,NaN,NaN,13.0,13.0,NaN,NaN,NaN,0.0,T,T,T,M2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,credit,325.0,87.0,NaN,NaN,gmail.com,NaN,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,M0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
sys.path.append(src_path)
from features.engineering import create_d_features
from about_data.split import temporal_split

json_path = os.path.join(data_path, r"processed/split_info.json")
with open(json_path, "r") as f:
    split_info = json.load(f)
    
train_end = split_info.get("train_end")
val_end = split_info.get("validation_end")

train, val, test = temporal_split(full_df, train_end, val_end)

In [6]:
map_dfs = {"train": train, "val": val, "test": test}
d_features_dfs = {}
for name, sample_df in map_dfs.items():
    d_features_dfs[name] = create_d_features(sample_df)

In [7]:
y_datasets = {}

for name, sample_df in map_dfs.items():
    y_datasets[name] = sample_df['isFraud']

In [8]:
X_train = d_features_dfs["train"].copy()
X_val = d_features_dfs["val"].copy()

y_train = y_datasets["train"]
y_val = y_datasets["val"]

In [9]:
models = {"xgboost": xgboost.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, objective="binary:logistic", eval_metric="aucpr", tree_method="hist", random_state=42, n_jobs=-1),
          "lightgbm": lightgbm.LGBMClassifier(n_estimators=300, learning_rate=0.05, objective="binary", metric='average_precision', is_unbalance=False, random_state=42, n_jobs=-1),
          "catboost": catboost.CatBoostClassifier(n_estimators=300, learning_rate=0.05, loss_function='Logloss', eval_metric='PRAUC', auto_class_weights='SqrtBalanced', random_state=42, thread_count=-1,)}

In [10]:
from model.preprocessor_pipe_evalueate import get_preprocessor, evaluate_model, create_pipeline

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment(experiment_name="fraud-detection-baseline")

results = []

for name, model in models.items():
    with mlflow.start_run(run_name=f"baseline {name}"):

        pipe = create_pipeline(model, get_preprocessor(X_train))
        
        pipe.fit(X_train, y_train)
        
        metrics = evaluate_model(pipe, X_val, y_val)
    
        mlflow.log_param("dataset", "D_features")
        
        mlflow.log_param("model", name)
    
        mlflow.log_param("feature_count", X_train.shape[1])
    
        mlflow.log_param("positive_rate_train", y_train.mean())
        
        mlflow.log_param("positive_rate_val", y_val.mean())
    
        mlflow.log_param("n_train", len(X_train))
    
        mlflow.log_metrics(metrics)
    
        results.append({**metrics})

🏃 View run baseline xgboost at: http://127.0.0.1:5000/#/experiments/1/runs/3344ea82df2849d28482c5eb0372069b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
[LightGBM] [Info] Number of positive: 14538, number of negative: 398840
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.077456 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 11659
[LightGBM] [Info] Number of data points in the train set: 413378, number of used features: 4098
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035169 -> initscore=-3.311794
[LightGBM] [Info] Start training from score -3.311794
🏃 View run baseline lightgbm at: http://127.0.0.1:5000/#/experiments/1/runs/61f7c607c0414e0893bbd05333ea4b6d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
0:	learn: 0.4428989	total: 383ms	remaining: 1m 54s
1:	learn: 0.4602548	total: 599ms	remaining: 1m 29

In [11]:
from features.engineering import create_d_no_aggregations_features
no_agg_d = {}
for name, d_df in map_dfs.items():
    no_agg_d[name] = create_d_no_aggregations_features(d_df)

In [ ]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment(experiment_name="fraud-detection-baseline")

results = []

X_train = no_agg_d['train']
X_val = no_agg_d['val']

for name, model in models.items():
    if name.startswith("cat"):
        pass

    with mlflow.start_run(run_name=f"{name}_no_agg"):
        pipe = create_pipeline(model, get_preprocessor(X_train))

        pipe.fit(X_train, y_train)
        
        metrics = evaluate_model(pipe, X_val, y_val)
    
        mlflow.log_param("dataset", "D_features_no_agg")
        
        mlflow.log_param("model", name)
    
        mlflow.log_param("feature_count", X_train.shape[1])
    
        mlflow.log_param("positive_rate_train", y_train.mean())
        
        mlflow.log_param("positive_rate_val", y_val.mean())
    
        mlflow.log_param("n_train", len(X_train))
    
        mlflow.log_metrics(metrics)
    
        results.append({**metrics})

🏃 View run xgboost_no_agg at: http://127.0.0.1:5000/#/experiments/1/runs/a95cc7cb20014975b7e3f8ce4114156d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
[LightGBM] [Info] Number of positive: 14538, number of negative: 398840
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.092709 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 11149
[LightGBM] [Info] Number of data points in the train set: 413378, number of used features: 4096
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035169 -> initscore=-3.311794
[LightGBM] [Info] Start training from score -3.311794
🏃 View run lightgbm_no_agg at: http://127.0.0.1:5000/#/experiments/1/runs/c58f7b9eff6b46e1a1df631e72883948
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
0:	learn: 0.4461490	total: 213ms	remaining: 1m 3s
1:	learn: 0.4608738	total: 410ms	remaining: 1m 1s
2:	l

In [ ]:
X_train.shape[1]